# 3. Reload the adapter and investigate failures

Use a fresh GPU runtime to prove the saved adapter is sufficient when combined with the pinned base model. You still need Hugging Face model access. Run the setup cell to download the tutorial source, install dependencies, and load your saved run from Drive.

[Open this lesson in Colab](https://colab.research.google.com/github/chenghungpan/llama3-qlora-finance-colab/blob/main/notebooks/03_reload_and_error_analysis.ipynb) · [GitHub repository](https://github.com/chenghungpan/llama3-qlora-finance-colab)


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

# The public repository is downloaded automatically. Clear this URL to upload a project ZIP instead.
REPO_URL = "https://github.com/chenghungpan/llama3-qlora-finance-colab.git"
PROJECT_NAME = "llama3-qlora-finance-colab"
if Path("finance_qlora/core.py").exists():
    ROOT = Path.cwd()
else:
    parent = Path("/content") if Path("/content").exists() else Path.cwd()
    ROOT = parent / PROJECT_NAME
    if not ROOT.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
        else:
            from google.colab import files
            print("Upload llama3-qlora-finance-colab.zip (the whole project, not just a notebook).")
            uploaded = files.upload()
            archive = next((name for name in uploaded if name.endswith(".zip")), None)
            if archive is None:
                raise ValueError("Please upload the project ZIP")
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (parent / member.filename).resolve()
                    if not target.is_relative_to(parent.resolve()):
                        raise ValueError("Unsafe archive path")
                z.extractall(parent)
    if not (ROOT / "finance_qlora/core.py").exists():
        raise ValueError("Expected the standalone project directory in the ZIP")
    os.chdir(ROOT)
print("Project directory:", ROOT)


In [ ]:
# Only run in a disposable Colab runtime or a dedicated environment.
# Vision/audio libraries are unused and may conflict with our pinned Torch version.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "-e", "."], check=True)
# ML commands below launch fresh Python processes, so they see the installed versions.


In [ ]:
# In Colab's left sidebar, open Secrets (key icon), add HF_TOKEN, and enable notebook access.
# Never paste a token into a cell or print it.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([sys.executable, "-m", "finance_qlora.preflight"], check=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/llama3-finance-runs/smoke-01")  # Change to your saved run.
assert (RUN_DIR / "adapter/tutorial_config.json").exists(), "Point RUN_DIR to the completed training run"


## Run a new example

The example below is invented for demonstration. Its output is a sentiment label, not a recommendation to buy or sell. Unexpected formatting is returned as `invalid`, not silently converted to a confident label.


In [ ]:
text = "The company raised its earnings outlook after stronger sales."
subprocess.run([sys.executable, "-m", "finance_qlora.predict", "--adapter", str(RUN_DIR / "adapter"), "--text", text], check=True)


## Analyze saved predictions without more GPU inference

Predictions contain IDs, labels, and generated answers. Reconstruct the pinned data to join source text for local analysis. The generated reports deliberately avoid copying entire source posts into telemetry.


In [ ]:
subprocess.run([sys.executable, "-m", "finance_qlora.data"], check=True)
from finance_qlora.core import read_jsonl
lookup = {r["id"]: r["text"] for r in read_jsonl("data/test.jsonl")}
predictions = read_jsonl(RUN_DIR / "finetuned/predictions.jsonl")
errors = [r for r in predictions if r["truth"] != r["prediction"]]
print(f"Errors: {len(errors)} / {len(predictions)}")
for row in errors[:10]:
    print({"text": lookup[row["id"]], "truth": row["truth"], "prediction": row["prediction"], "generated": row["generated_text"]})


## Exercises and limits

1. Categorize errors: ambiguity, sarcasm, mixed entities, missing context, or output format.
2. Compare class recall before and after training using `comparison.json`.
3. Design a validation-only experiment changing rank or training size while holding other settings fixed.
4. Explain why a bullish post can precede a falling price.

This dataset lacks the verified event timestamps, tradable-security mapping, and aligned future returns needed for a valid price-forecasting experiment. See `docs/forecasting_extension.md` for a separate research design. Do not treat these sentiment scores as evidence of trading profitability.

Before sharing a trained adapter, review Meta's model license, include required attribution/notice, and check the dataset's source terms. Code licensing does not relicense model weights or source posts. Disconnect the GPU when finished.
